# Sistema de Q&A con Routing entre SQL y Documentos (RAG)

Este notebook implementa un sistema de preguntas y respuestas capaz de responder consultas a partir de dos fuentes de información:

- Una base de datos relacional consultada mediante SQL
- Un conjunto de documentos de texto consultados mediante búsqueda semántica

El sistema utiliza embeddings para recuperar documentos relevantes y un mecanismo de routing para decidir automáticamente si una pregunta debe resolverse mediante SQL o mediante recuperación de documentos.

## 1 - Importo Librerías

In [ ]:
import os #para navegar los documentos
import sqlite3 #para importar la base SQL
import pandas as pd #para leer/trabajar los datos SQL
import numpy as np #para calculos

from sentence_transformers import SentenceTransformer #para embeddings
from sklearn.metrics.pairwise import cosine_similarity #para el calculo de similaridad de coseno en los embeddings
import warnings #para ignorar alertas a la hora de ejecutar la notebook
warnings.filterwarnings('ignore')

## 2 - Conexión Base SQL y exploracion de la base de datos.

In [ ]:
conn = sqlite3.connect("tienda.db")

2.1 Vemos que tablas hay

In [ ]:
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
tables

2.2 Investigo que hay dentro de las tablas

Productos

In [ ]:
pd.read_sql("SELECT * FROM productos;", conn)

*Veo cuántas filas tiene la tabla*

In [ ]:
pd.read_sql("SELECT count(*) FROM productos;", conn)

Clientes

In [ ]:
pd.read_sql("SELECT * FROM clientes;", conn)

*Veo cuántas filas tiene la tabla*

In [ ]:
pd.read_sql("SELECT count(*) FROM clientes;", conn)

Pedidos

In [ ]:
pd.read_sql("SELECT * FROM pedidos;", conn)

*Veo cuántas filas tiene la tabla*

In [ ]:
pd.read_sql("SELECT count(*) as q_pedidos FROM pedidos;", conn)

#### 2.3 - Consultas a las bases

*¿Cuántas compras se hicieron por periodo y cuánto se gastó?*

In [ ]:
pd.read_sql("SELECT (substr(fecha,1,4)*100 + substr(fecha,6,2)) as periodo, count(*) as q_pedidos, sum (total) as gasto_total FROM pedidos group by 1 order by periodo;", conn)

En octubre fue el mes que más compras hubo y que más dineró se gastó.

*¿Cuantos pedidos hizo cada cliente y cuánta plata gastó?*

In [ ]:
pd.read_sql("SELECT cl.nombre, count(*) as q_pedidos, sum (total) as gasto_total FROM pedidos as pd LEFT JOIN clientes cl on pd.cliente_id = cl.id group by 1 order by q_pedidos desc;", conn)

Carlos López es el cliente que más compró y también el que más gastó.

*¿Cuantos stock tenemos por categoría?*

In [ ]:
pd.read_sql("SELECT categoria, sum(stock) as stock FROM productos group by 1 order by stock desc;", conn)

En electrónicos es donde más stock tenemos.

*¿Es electrónica también la categoría en la que mayor dinero hay invertido?*

In [ ]:
pd.read_sql("SELECT categoria, sum(stock*precio) as inversion_total FROM productos group by 1 order by inversion_total desc;", conn)

Efectivamente, también es en donde más dinero si invirtió.

## 3 - Cargar documentos de texto

In [ ]:
documents = {} #creo un diccionario

folder_path = "/content" #camino en colab

for file in os.listdir(folder_path):
    if file.endswith(".txt"):
        with open(os.path.join(folder_path, file), "r", encoding="utf-8") as f:
            documents[file] = f.read()

for key in documents:
  print(f"{key} archivo cargado")

*Valido el contenido de los documentos*

In [ ]:
for key, value in documents.items():
  print(f"{key}:\n\n{value}")


El contenido se ve correctamente cargado.

*Reviso la cantidad de caracteres por archivo para el embedding.*

In [ ]:
for name, text in documents.items():
    print(name, "→", len(text), "caracteres")

## 4 - Crear Embeddings

Para poder realizar búsqueda semántica, convertimos cada documento en un embedding utilizando un modelo de sentence-transformers. Esto permite representar el significado del texto en un espacio vectorial.

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Elijo este modelo por que es chico y rápido para la tarea que tenemos que ejecutar dados los datos que tenemos.

#### *Creo el embedding*

In [ ]:
document_embeddings = {}

for name, text in documents.items():
    embedding = model.encode(text)
    document_embeddings[name] = embedding

In [ ]:
for keys, values in document_embeddings.items():
  print(f"{keys}: {values[:5]}\n")

Los vectores fueron creados.

Valido el tamaño de los vectores:

In [ ]:
for name, emb in document_embeddings.items():
    print(name, "→ tamaño del vector:", len(emb))

## 5 -  Implemento búsqueda semántica

In [ ]:
def busqueda_documentos(question):

    question_embedding = model.encode(question)

    best_doc = None
    best_score = -1

    for name, emb in document_embeddings.items():
        score = cosine_similarity(
            [question_embedding],
            [emb]
        )[0][0]

        if score > best_score:
            best_score = score
            best_doc = name

    return best_doc, best_score

Caso de prueba:

In [ ]:
busqueda_documentos("cómo me contacto con la tienda?")

Mejoramos display:

In [ ]:
question = input("Escribe tu pregunta: ")

result = busqueda_documentos(question)

print("\nDocumento encontrado:", result[0])

In [ ]:
while True:

    question = input("\nPregunta (escribe 'salir' para terminar): ")

    if question.lower() == "salir":
        break

    result = busqueda_documentos(question)

    print("\nDocumento encontrado:", result[0])
    print("Similitud:", result[1])

Responde algunas consultas bien, pero la gran mayoría mal. En este caso las preguntas respondidas correctamente fueron 2/9 **~22%**.

Voy a probar cambiando de modelo a alguno que esté más relacionado al español ya que el utilizado está optimizado para inglés.

También, para darle algo más de contexto al modelo, voy a agregar los títulos de los documentos al embedding.

In [ ]:
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

Nuevo modelo corrido.

In [ ]:
document_embeddings = {}

for name, text in documents.items():
    enriched_text = f"Documento sobre {name.replace('.txt','')}: {text}"
    embedding = model.encode(enriched_text)
    document_embeddings[name] = embedding

Contexto agregado.

*La función previamente creada para utilizar la similitud de coseno sigue funcionando porque reemplacé los objetos pero no el nombre del objeto.*

In [ ]:
chat_history = []

Guardo las preguntas y respuestas en una lista para después comparar. También, directamente creo la función de preguntas/respuestas con su guardado.

In [ ]:
def chat_QA():

    while True:

        question = input("\nPregunta (escribe 'salir' para terminar): ")

        if question.lower() == "salir":
            break

        doc, score = busqueda_documentos(question)

        print("\nDocumento encontrado:", doc)
        print("Similitud:", score)

        feedback = input("\n¿La respuesta fue útil? (si/no): ").lower()

        chat_history.append({
            "question": question,
            "document": doc,
            "score": score,
            "feedback": feedback
        })

    return pd.DataFrame(chat_history)['feedback'].value_counts(normalize=True) * 100

In [ ]:
chat_QA()

In [ ]:
pd.DataFrame(chat_history)['feedback'].value_counts(normalize=True) * 100

Vemos una amplia mejora al **~67%** con los cambios efectuados. Nos quedamos con el modelo multilingual.

Para evitar tener que escribir las preguntas 1 a 1, dejo también la opción de mandar todas juntas:

In [ ]:
questions = [
    "cuando llega mi producto?",
    "que garantía tiene mi compra?",
    "que garantía tiene un mueble?",
    "los accesorios tienen garantía?",
    "cuanto tarda un envío?",
    "quien es el transportista?",
    "en cuanto me reembolsan?",
    "cuanto hay que gastar para un envio gratis?",
    "cuanto tarda la entrega?"
]

In [ ]:
resultados = []

for q in questions:

    doc, score = busqueda_documentos(q)

    resultados.append({
        "pregunta": q,
        "documento": doc,
        "score": score
    })

In [ ]:
pd.DataFrame(resultados)

Lo guardamos para futuras mejoras, sobre todo, para validar las preguntas finales en caso de que no sean bien respondidas.

 ## 6 - Sistema de Q&A (routing) + Preguntas de prueba

Voy a hacer dos versiones, una "barata" con palabras más relacionadas a SQL y otro donde todas las preguntas pasan por el proceso de embedding, más "caro"/"costoso" computacionalmente, pero con mejor experiencia para el usuario.

#### 6.1 Modelo Simple

Ejemplos de palabras para sql

In [ ]:
palabras_sql = [
    "cuántos",
    "cuantas",
    "cantidad",
    "total",
    "clientes",
    "pedidos",
    "compras",
    "stock",
    "categoría",
    "ventas",
    "gastó",
    "gasto"
]

Armo enrutador

In [ ]:
def enrutador_preguntas(pregunta):

    q = pregunta.lower()

    for word in palabras_sql:
        if word in q:
            return "sql"

    return "documentos"

Función para preguntas relacionadas a SQL

In [ ]:
def respuesta_sql(pregunta):

    if "clientes" in pregunta.lower():
        query = "SELECT COUNT(*) FROM clientes"
        result = pd.read_sql(query, conn)
        respuesta = f"Tenemos {result.iloc[0,0]} clientes."

    elif "stock por categoria" in pregunta.lower():
        query = """
        SELECT categoria, SUM(stock) as stock_total
        FROM productos
        GROUP BY 1
        """
        result = pd.read_sql(query, conn)
        respuesta = result.to_string(index=False)

    elif "cuantos pedidos" in pregunta.lower() or "cuántos pedidos" in pregunta.lower():
        query = """
        SELECT count(distinct id) as pedidos_totales FROM pedidos
        """
        result = pd.read_sql(query, conn)
        respuesta = f"Tenemos {result.iloc[0,0]} pedidos."

    else:
        respuesta = "No tengo una consulta SQL definida para esa pregunta."

    return respuesta, "SQL"

Función para preguntas relacionadas a documentos

In [ ]:
def respuesta_documentos(pregunta):

    doc, score = busqueda_documentos(pregunta)

    respuesta = documents.get(doc, "")

    return respuesta, doc

Sistema Final QA:

In [ ]:
def sistema_QA(pregunta):

    route = enrutador_preguntas(pregunta)

    if route == "sql":
        answer, source = respuesta_sql(pregunta)

    else:
        answer, source = respuesta_documentos(pregunta)

    return {
        "pregunta": pregunta,
        "respuesta": answer,
        "fuente": source
    }

Pruebas con preguntas de ejemplo

In [ ]:
sistema_QA("Cuántos clientes tenemos?")

In [ ]:
sistema_QA("Cuánto cuesta el envío nacional?")

In [ ]:
sistema_QA("Cuantos pedidos hay en total?")

In [ ]:
sistema_QA("Qué garantía tienen los productos?")

Guardo todo en una lista para hacer pruebas y luego poder verlas de mejor manera

In [ ]:
chat_history_v2 = []

In [ ]:
def chat_QA_v2():

    while True:

      pregunta = input("\nPregunta (escribe 'salir'): ")

      if pregunta.lower() == "salir":
          break

      resultado = sistema_QA(pregunta)

      print("\nRespuesta:")
      print(resultado["respuesta"])

      print("\nFuente:", resultado["fuente"])

      feedback = input("\n¿La respuesta fue útil? (si/no): ").lower()

      chat_history_v2.append({
            "pregunta": pregunta,
            "respuesta": resultado["respuesta"],
            "fuente":resultado["fuente"],
            "feedback": feedback
        })

Armamos un chatbot para iterar las preguntas y respuestas con feedback

In [ ]:
chat_QA_v2()

Vemos que las preguntas y respuestas hechas son correctamente canalizadas y respondidas.

#### 6.2 Modelo Sofisticado

Ejemplos de preguntas para preparar el modelo

In [ ]:
ejemplos_sql = [
    "¿Cuántos clientes tenemos?",
    "¿Cuántos pedidos se hicieron?",
    "¿Cuál es el total de ventas?",
    "¿Cuánto stock hay por categoría?",
    "¿Cuánto gastó cada cliente?"
]

ejemplos_doc = [
    "¿Cómo puedo devolver un producto?",
    "¿Cuánto tarda el envío?",
    "¿Cómo contacto al soporte?",
    "¿Cuál es la política de devoluciones?",
    "¿Quién realiza el envío?",
    "¿Qué garantía tienen los productos?"
]

Preparo embeddings de los ejemplos

In [ ]:
sql_embeddings = model.encode(ejemplos_sql)
doc_embeddings = model.encode(ejemplos_doc)

Enrutador con pregunta con embedding

In [ ]:
def enrutador_con_embedding(pregunta):

    p_emb = model.encode(pregunta)

    sql_score = cosine_similarity([p_emb], sql_embeddings).max()
    doc_score = cosine_similarity([p_emb], doc_embeddings).max()

    if sql_score > doc_score:
        return "SQL"
    else:
        return "Documents"

Pruebo el enrutador

In [ ]:
enrutador_con_embedding("¿Cuántos clientes tenemos?")

In [ ]:
enrutador_con_embedding("¿Qué garantía tienen los productos?")

In [ ]:
enrutador_con_embedding("¿Qué garantía tienen los muebles?")

In [ ]:
enrutador_con_embedding("¿Hay algún chat de contacto?")

funciona OK

In [ ]:
def sistema_QA_v2(pregunta):

    ruta = enrutador_con_embedding(pregunta)

    if ruta == "SQL":
        respuesta, fuente = respuesta_sql(pregunta)

    else:
        respuesta, fuente = respuesta_documentos(pregunta)

    return {
        "pregunta": pregunta,
        "respuesta": respuesta,
        "fuente": fuente,
        "ruta": ruta
    }

Pruebas

In [ ]:
sistema_QA_v2("Cuántos clientes tenemos?")

In [ ]:
sistema_QA_v2("Cuánto cuesta el envío nacional?")

In [ ]:
sistema_QA_v2("Cuantos pedidos hay en total?")

In [ ]:
sistema_QA_v2("Qué garantía tienen los productos?")

Guardo todo en una lista para hacer pruebas y luego poder verlas de mejor manera

In [ ]:
chat_history_v3 = []

In [ ]:
def chat_QA_v3():
    while True:

      pregunta = input("\nPregunta (escribe 'salir'): ")

      if pregunta.lower() == "salir":
          break

      resultado = sistema_QA_v2(pregunta)

      print("\nRespuesta:")
      print(resultado["respuesta"])

      print("\nFuente:", resultado["fuente"])
      print("\nRuta:", resultado["ruta"])

      feedback = input("\n¿La respuesta fue útil? (si/no): ").lower()

      chat_history_v3.append({
            "pregunta": pregunta,
            "respuesta": resultado["respuesta"],
            "fuente":resultado["fuente"],
            "ruta": resultado["ruta"],
            "feedback": feedback
        })

Armamos un chatbot para iterar las preguntas y respuestas con feedback

In [ ]:
chat_QA_v3()

Dependiendo las necesidades de negocio que tengamos, tal vez este modelo no es tan necesario ya que el anterior responde bien las consultas.

De todas formas, las preguntas realizadas, ya están contempladas previamente por la base de SQL. Si se busca responder algo por fuera de eso, sería mejor mantener un modelo como este y/o evolucionarlo a alguno que permita transformar las preguntas de lenguaje natural en sintáxis SQL.


----------------------------------------------------------------------------

Más allá de este pequeño comentario, este modelo responde correctamente todas las preguntas. Pero me gustaría ir un poco más allá y que responda solamente la parte del documento relacionada a la pregunta. Para eso vamos a utilizar chunks en los documentos (.txt), y en lugar de recuperar el documento completo, el sistema realiza la búsqueda semántica sobre estos chunks y devuelve el más relevante para la pregunta del usuario.

#### 6.3 Modelo con Chunks

Dividimos al documento en chunks (fragmentos):

In [ ]:
chunks = []
chunk_docs = []

for name, text in documents.items():

    oraciones = text.split("\n") #usamos el salto de línea por como están hechos los documentos

    for s in oraciones:
        chunks.append(s)
        chunk_docs.append(name)

Veo como quedan los chunks

In [ ]:
chunks

Creo embeddings de los chunks

In [ ]:
chunk_embeddings = model.encode(chunks)

Ahora cada frase tiene su vector asociado

Busco el fragmento más relevante

In [ ]:
def search_chunks(pregunta):

    q_emb = model.encode(pregunta)

    scores = cosine_similarity([q_emb], chunk_embeddings)[0]

    best_idx = scores.argmax()

    return chunks[best_idx], chunk_docs[best_idx], scores[best_idx]

Probamos el modelo:

In [ ]:
chunk, source, score = search_chunks("¿Cuánto tarda el envío?")

print("Respuesta:", chunk)
print("Fuente:", source)

In [ ]:
chunk, source, score = search_chunks("¿Cuánto cuesta el envío nacional?")

print("Respuesta:", chunk)
print("Fuente:", source)

In [ ]:
chunk, source, score = search_chunks("¿Puedo devolver un producto?")

print("Respuesta:", chunk)
print("Fuente:", source)

In [ ]:
chunk, source, score = search_chunks("¿Qué garantía tienen los productos?")

print("Respuesta:", chunk)
print("Fuente:", source)

Algunas respuestas están bien, pero a otras se les nota que les falta contexto.

Vamos a rehacer el algoritmo con el nombre del documento + fragmentos de al menos 60 caracteres. Elijo esa cantidad de caracteres en base al conteo del punto 4 cuando hicimos los embeddings sobre los documentos.

In [ ]:
chunks = []
chunk_docs = []

tamanio_chunk = 60

for name, text in documents.items():

    for i in range(0, len(text), tamanio_chunk):

        chunk = text[i:i+tamanio_chunk]

        # contexto agregado
        enriched_chunk = f"Documento sobre {name.replace('.txt','')}: {chunk}"

        chunks.append(enriched_chunk)
        chunk_docs.append(name)

Vemos como queda el chunk:

In [ ]:
chunks

Volvemos a hacer el modelo

In [ ]:
chunk_embeddings = model.encode(chunks)

In [ ]:
def search_chunks(question):

    q_emb = model.encode(question)

    scores = cosine_similarity([q_emb], chunk_embeddings)[0]

    best_idx = scores.argmax()

    return chunks[best_idx], chunk_docs[best_idx], scores[best_idx]

Limpiamos respuesta y probamos

In [ ]:
chunk, source, score = search_chunks("¿Cuánto cuesta el envío nacional?")

clean_answer = chunk.split(":", 1)[1].strip()

print("Respuesta:", clean_answer)
print("Fuente:", source)
print("Similitud:", score)

In [ ]:
chunk, source, score = search_chunks("¿Puedo devolver un producto?")

clean_answer = chunk.split(":", 1)[1].strip()

print("Respuesta:", clean_answer)
print("Fuente:", source)
print("Similitud:", score)

In [ ]:
chunk, source, score = search_chunks("¿Qué garantía tienen los productos?")

clean_answer = chunk.split(":", 1)[1].strip()

print("Respuesta:", clean_answer)
print("Fuente:", source)
print("Similitud:", score)

- Vemos que si bien mejora el contexto, las respuestas no son del todo precisas y hasta algunas quedan cortadas.

- Entendemos que la experiencia sería mejor si se pule el modelo, pero el limitante de que por cada pregunta haya que limitarse a "x" caracteres, hace que la tecnología chunk no sea tan performante bajo las características de este ejercicio.

Hacemos un último intento, **chunkeando por 3 oraciones seguidas** (en este caso por salto de línea) y no por caracteres, para no perder el significado.

In [ ]:
chunks = []
chunk_docs = []

for name, text in documents.items():

    sentences = text.split("\n")

    for i in range(0, len(sentences), 2): #línea de 3 oraciones seguidas

      chunk = " ".join(sentences[i:i+2])

      enriched_chunk = f"Documento sobre {name.replace('.txt','')}: {chunk}"

      chunks.append(enriched_chunk)
      chunk_docs.append(name)

Veo como quedan los chunks:

In [ ]:
chunks

Reproceso el modelo con las nuevas condiciones

In [ ]:
chunk_embeddings = model.encode(chunks)

Valido que las tres partes tengan los mismos chunks.

In [ ]:
print("chunks:", len(chunks))
print("chunk_docs:", len(chunk_docs))
print("embeddings:", len(chunk_embeddings))

Reproceso la función

In [ ]:
def search_chunks(pregunta):

    q_emb = model.encode(pregunta)

    scores = cosine_similarity([q_emb], chunk_embeddings)[0]

    best_idx = scores.argmax()

    return chunks[best_idx], chunk_docs[best_idx], scores[best_idx]

Limpiamos respuesta y probamos

In [ ]:
chunk, source, score = search_chunks("¿Puedo devolver un producto?")

clean_answer = chunk.split(":", 1)[1].strip()

print("Respuesta:", clean_answer)
print("Fuente:", source)
print("Similitud:", score)

Generamos el sistema para probar con las tres preguntas que deberían ser respondidas por esta parte del RAG:

In [ ]:
questions = [
    "¿Cuánto cuesta el envío nacional?",
    "¿Puedo devolver un producto?",
    "¿Qué garantía tienen los productos?"
]

for q in questions:
    chunk, source, score = search_chunks(q)

    clean_answer = chunk.split(":", 1)[1].strip()

    print("\nPregunta:", q)
    print("Respuesta:", clean_answer)
    print("Fuente:", source)
    print("Similitud:", score)

- No termina de responder bien el modelo a pesar de la iteración. Aunque la similitud de coseno incrementó.

Con los resultados expuestos, vamos a quedarnos con el modelo del punto 6.2, que es el modelo con mayores posibilidades de mejorar.

## 7- Conclusión

En este ejercicio se desarrolló un sistema de **preguntas y respuestas (Q&A)** capaz de responder consultas utilizando dos fuentes de información diferentes: una **base de datos estructurada** y un **conjunto de documentos de texto**.

El flujo parte de la carga y limpieza de los documentos, luego se generan embeddings y finalmente se implementa un sistema que recupera el contenido más relevante para responder preguntas del usuario.

Se exploraron distintas aproximaciones de recuperación. La versión final utiliza embeddings sobre documentos completos, lo que permitió obtener respuestas más estables y mantener un pipeline claro:
pregunta → embedding → búsqueda por similitud → generación de respuesta con contexto.

También se experimentó con chunking de documentos para mejorar la granularidad de la recuperación. Sin embargo, en las pruebas realizadas esta estrategia no produjo respuestas suficientemente coherentes, por lo que no se incorporó en la versión final del sistema.

Como posibles mejoras futuras, se podrían considerar:

- utilizar un modelo de clasificación más robusto para el routing

- incorporar un modelo generativo (LLM) para producir respuestas más naturales

- almacenar los embeddings en una base de datos vectorial para mejorar la escalabilidad

- incorporar un text-to-SQL que traduzca lenguaje natural a queries reales automáticamente.